## Определение влияния возраста на содержание Ig A

Для включения категориального фактора (номера группы) введем $p=5$ индикаторных переменных: $x_1, \dots, x_5$. Принадлежность пациента к $j$-й группе означает $x_j = 1$, остальные нули.
$\sum_{i=1}^5 x_i = 1$.
Во избежание вырожденности матрицы Фишера свободный член ($\beta_0$) не вводится. Уравнение регрессии:
$$\eta = \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_3 + \beta_4 x_4 + \beta_5 x_5 + \varepsilon$$

Матрица наблюдений $\Psi$ имеет размер $n \times p$ ($25 \times 5$).
Информационная матрица $F = \Psi^T \Psi$ строго диагональна: $F = \text{diag}(n_1, n_2, n_3, n_4, n_5)$.
Вектор оценок вычисляется как: $\tilde{\beta} = F^{-1}\Psi^T Y$.

**Проверка гипотез**
* **О значимости коэффициентов ($H_0: \beta_i = 0$):**
  Используется статистика Стьюдента: $\Delta = \frac{\tilde{\beta}_i}{\sqrt{RSS \cdot F^{-1}_{ii}}} \sqrt{n-p} \sim t(n-p)$
* **О значимости регрессии в целом ($H_0:$ все $\beta_i = 0$):**
  Это проверка влияния возраста на уровень $Ig A$. Используется статистика Фишера:
  $$\Delta = \frac{R^2}{1-R^2} \cdot \frac{n-p}{p-1} \sim F(p-1, n-p)$$
  Если $p\text{-value} < 0.05$, регрессия значима (возраст влияет на отклик).

In [11]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import itertools

In [12]:
data = {
    1: [83, 85], 2: [84, 85, 85, 86, 86, 87],
    3: [86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90],
    4: [89, 90, 90, 91], 5: [90, 92]
}

df = pd.DataFrame([(g, v) for g, vals in data.items() for v in vals], columns=['Group', 'Ig_A'])
n, p = len(df), len(data)

display(df)

# pd.get_dummies автоматически создает матрицу индикаторов
Psi = pd.get_dummies(df['Group'], dtype=int).values
Y = df['Ig_A'].values

F = Psi.T @ Psi
F_inv = np.linalg.inv(F)
beta_tilde = F_inv @ Psi.T @ Y

print("F:\n", F, "\n")

RSS = np.sum((Y - Psi @ beta_tilde)**2)
TSS = np.sum((Y - np.mean(Y))**2)
R2 = (TSS - RSS) / TSS

# проверка значимости коэффициентов
df_t = n - p
t_stats = beta_tilde / np.sqrt(RSS * np.diag(F_inv)) * np.sqrt(df_t)
p_vals_t = 2 * stats.t.sf(np.abs(t_stats), df_t)

summary_df = pd.DataFrame({
    'β̃': beta_tilde,
    'Δ': t_stats,
    'p-value': p_vals_t
}, index=[f'β{i}' for i in range(1, p + 1)])

summary_df['Вывод'] = np.where(summary_df['p-value'] < 0.05, 'Значим', 'Незначим')
print("1) Проверка значимости коэффициентов")
display(summary_df.round(4))

print("2) Проверка значимости регрессии в целом")
# проверка значимости регрессии в целом
df1_f = p - 1

delta_F = (R2 / (1 - R2)) * (df_t / df1_f)
p_val_F = stats.f.sf(delta_F, df1_f, df_t)

print(f"RSS = {RSS:.4f}, TSS = {TSS:.4f}, R² = {R2:.4f}")
print(f"Δ = {delta_F:.4f}, p-value = {p_val_F:.2e}\n")

if p_val_F < 0.05:
    print("p-value < 0.05. Регрессия значима")
    print("Возраст оказывает статистически значимое влияние на содержание Ig A в крови.")
else:
    print("p-value >= 0.05. Регрессия незначима")
    print("Нет оснований утверждать, что возраст влияет на содержание Ig A.")

,Group,Ig_A
0,1,83
1,1,85
2,2,84
3,2,85
4,2,85
5,2,86
6,2,86
7,2,87
8,3,86
9,3,87


F:
 [[ 2  0  0  0  0]
 [ 0  6  0  0  0]
 [ 0  0 11  0  0]
 [ 0  0  0  4  0]
 [ 0  0  0  0  2]] 

1) Проверка значимости коэффициентов


,β̃,Δ,p-value,Вывод
β1,84.0000,110.4490,0.0,Значим
β2,85.5000,194.7194,0.0,Значим
β3,87.8182,270.7997,0.0,Значим
β4,90.0000,167.3555,0.0,Значим
β5,91.0000,119.6531,0.0,Значим


2) Проверка значимости регрессии в целом
RSS = 23.1364, TSS = 122.1600, R² = 0.8106
Δ = 21.4000, p-value = 5.41e-07

p-value < 0.05. Регрессия значима
Возраст оказывает статистически значимое влияние на содержание Ig A в крови.


## Попарное сравнение средних (Поправка Холма-Бонферрони)

Для каждой пары групп ($i$ и $j$) проверяется гипотеза о равенстве их истинных средних:
* $H_0: \beta_i = \beta_j$
* $H_1: \beta_i \neq \beta_j$

Количество таких проверок: $m = \frac{p(p-1)}{2} = 10$.
Статистика Стьюдента для проверки разности двух коэффициентов в нашей модели:
$$\Delta = \frac{\tilde{\beta}_i - \tilde{\beta}_j}{\sqrt{RSS \left(F^{-1}_{ii} + F^{-1}_{jj}\right)}} \sqrt{n-p} \sim t(n-p)$$

Алгоритм поправки:
1. Вычисленные $p\text{-value}$ сортируются по возрастанию: $p_{(1)} \le p_{(2)} \le \dots \le p_{(10)}$.
2. На шаге $k$ значение $p_{(k)}$ сравнивается с индивидуальным порогом: $\alpha_k = \frac{\alpha}{m - k + 1}$.
3. **Процедура остановки:** Если $p_{(k)} < \alpha_k$, гипотеза $H_0$ отвергается (различие признается значимым). Как только достигается первый шаг, где $p_{(k)} \ge \alpha_k$, гипотеза принимается, и **все последующие гипотезы также автоматически принимаются**, процедура останавливается.

In [15]:
alpha = 0.05
m = p * (p - 1) // 2

# считаем статистики и p-value для всех пар
results = []
for i, j in itertools.combinations(range(p), 2):
    diff = beta_tilde[i] - beta_tilde[j]
    se = np.sqrt(RSS * (F_inv[i, i] + F_inv[j, j])) / np.sqrt(df_t)

    t_stat = diff / se
    p_val = 2 * stats.t.sf(np.abs(t_stat), df_t)

    results.append({
        'Пара': f'Гр. {i+1} и Гр. {j+1}',
        '|Δβ|': np.abs(diff),
        'Δ': t_stat,
        'p-value': p_val
    })

# сортируем p-value по возрастанию
df_pairs = pd.DataFrame(results).sort_values('p-value').reset_index(drop=True)

# применяем поправку
df_pairs['Шаг (k)'] = df_pairs.index + 1
df_pairs['Порог α_k'] = alpha / (m - df_pairs['Шаг (k)'] + 1)

decisions = []
reject_flag = True # флаг для остановки процедуры

for _, row in df_pairs.iterrows():
    # если мы еще не остановились и p-value пробивает порог
    if reject_flag and row['p-value'] < row['Порог α_k']:
        decisions.append('Отвергаем H0 (Различаются)')
    else:
        reject_flag = False # процедура остановлена, дальше всё принимается
        decisions.append('Принимаем H0 (Не различаются)')

df_pairs['Вывод'] = decisions

cols = ['Шаг (k)', 'Пара', '|Δβ|', 'Δ', 'p-value', 'Порог α_k', 'Вывод']
display(df_pairs[cols].round(4))

,Шаг (k),Пара,|Δβ|,Δ,p-value,Порог α_k,Вывод
0,1,Гр. 1 и Гр. 5,7.0000,-6.5083,0.0000,0.0050,Отвергаем H0 (Различаются)
1,2,Гр. 2 и Гр. 4,4.5000,-6.4817,0.0000,0.0056,Отвергаем H0 (Различаются)
2,3,Гр. 1 и Гр. 4,6.0000,-6.4415,0.0000,0.0062,Отвергаем H0 (Различаются)
3,4,Гр. 2 и Гр. 5,5.5000,-6.2629,0.0000,0.0071,Отвергаем H0 (Различаются)
4,5,Гр. 1 и Гр. 3,3.8182,-4.6181,0.0002,0.0083,Отвергаем H0 (Различаются)
5,6,Гр. 2 и Гр. 3,2.3182,-4.2468,0.0004,0.0100,Отвергаем H0 (Различаются)
6,7,Гр. 3 и Гр. 5,3.1818,-3.8484,0.0010,0.0125,Отвергаем H0 (Различаются)
7,8,Гр. 3 и Гр. 4,2.1818,-3.4743,0.0024,0.0167,Отвергаем H0 (Различаются)
8,9,Гр. 1 и Гр. 2,1.5000,-1.7081,0.1031,0.0250,Принимаем H0 (Не различаются)
9,10,Гр. 4 и Гр. 5,1.0000,-1.0736,0.2958,0.0500,Принимаем H0 (Не различаются)
